# 13 -- Public Score Review (Contact Luck v0.12)

Freezes Versions 0.2-0.11 completely. This notebook inspects the already
-computed outputs of the public score contract:

  - `mlb_luck_score.scoring.public_score_schema` -- the versioned public
    player-season schema.
  - `mlb_luck_score.scoring.public_score_table` -- schema population from the
    frozen Version 0.11 aggregation (named `public_score_table.py`, not
    `public_score.py` -- that name is already taken by the live, tested
    Version 0.1 legacy score mapping).
  - `mlb_luck_score.scoring.leaderboard` -- competition-ranked, qualified-only
    leaderboards (favorable and least-favorable).
  - `mlb_luck_score.scoring.public_labels` -- centralized public language.

The OFFICIAL public metric is **Contact Luck Runs per 100 Eligible Batted
Balls**. It is retrospective: a description of realized outcomes, not a
projection of future performance and not a measure of stable batting talent.

Run `make run-public-score` to (re)generate the JSON/CSV/Parquet outputs this
notebook reads. No cell here re-runs model training.

In [ ]:
import json
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

TABLES_DIR = Path("../outputs/tables")

TABLE_PATH = TABLES_DIR / "public_score_v012.json"
FAVORABLE_PATH = TABLES_DIR / "public_score_v012_favorable_leaderboard.json"
UNFAVORABLE_PATH = TABLES_DIR / "public_score_v012_unfavorable_leaderboard.json"
SCORECARD_PATH = TABLES_DIR / "public_score_v012_scorecard.json"
REVIEW_PATH = TABLES_DIR / "public_score_v012_review_tables.json"

public_score_table = pd.read_json(TABLE_PATH) if TABLE_PATH.exists() else None
favorable_leaderboard = pd.read_json(FAVORABLE_PATH) if FAVORABLE_PATH.exists() else None
unfavorable_leaderboard = pd.read_json(UNFAVORABLE_PATH) if UNFAVORABLE_PATH.exists() else None
score_card = json.loads(SCORECARD_PATH.read_text()) if SCORECARD_PATH.exists() else None
review_tables = json.loads(REVIEW_PATH.read_text()) if REVIEW_PATH.exists() else None

if public_score_table is not None:
    print(f"Loaded {len(public_score_table):,} public player-season rows")
else:
    print("public_score_v012.json not found -- run `make run-public-score` first")

## 1. Score card

Compact model/score metadata -- version, generation time, data-through date,
qualification counts, ranking/interval method.

In [ ]:
if score_card is not None:
    print(json.dumps(score_card, indent=2))

## 2. Public language (Phase 6)

Centralized definitions -- these exact strings are what any public-facing
surface must use.

In [ ]:
from mlb_luck_score.scoring.public_labels import (
    CONTACT_LUCK_RUNS_LABEL,
    CONTACT_LUCK_RUNS_DEFINITION,
    CONTACT_LUCK_RUNS_PER_100_LABEL,
    CONTACT_LUCK_RUNS_PER_100_DEFINITION,
    POSITIVE_VALUE_DEFINITION,
    NEGATIVE_VALUE_DEFINITION,
    RETROSPECTIVE_LIMITATION,
    MOST_FAVORABLE_LEADERBOARD_LABEL,
    LEAST_FAVORABLE_LEADERBOARD_LABEL,
)

print(f"{CONTACT_LUCK_RUNS_LABEL}: {CONTACT_LUCK_RUNS_DEFINITION}")
print(f"{CONTACT_LUCK_RUNS_PER_100_LABEL}: {CONTACT_LUCK_RUNS_PER_100_DEFINITION}")
print(f"Positive value: {POSITIVE_VALUE_DEFINITION}")
print(f"Negative value: {NEGATIVE_VALUE_DEFINITION}")
print(f"\nLimitation: {RETROSPECTIVE_LIMITATION}")
print(f"\nLeaderboards: '{MOST_FAVORABLE_LEADERBOARD_LABEL}' / '{LEAST_FAVORABLE_LEADERBOARD_LABEL}'")

## 3. Most favorable qualified leaderboard

Highest Contact Luck Runs per 100, qualified rows only, competition ranking
(ties share a rank; the next distinct value skips ahead).

In [ ]:
if favorable_leaderboard is not None:
    display(favorable_leaderboard.head(25))

## 4. Least favorable qualified leaderboard

"Least favorable outcomes relative to expectation" -- never described as
"worst players." Lowest Contact Luck Runs per 100, qualified rows only.

In [ ]:
if unfavorable_leaderboard is not None:
    display(unfavorable_leaderboard.head(25))

## 5. Interval presentation (Phase 3)

Point estimate and 95% game_pk-clustered sampling interval are ALWAYS shown
together, plus sample size and games. `interval_interpretation` is
descriptive only -- never a significance or skill claim.

In [ ]:
if public_score_table is not None:
    qualified = public_score_table[public_score_table["qualification_status"] == "qualified"]
    display(
        qualified[
            [
                "batter_id",
                "contact_luck_runs_per_100",
                "lower_95_interval",
                "upper_95_interval",
                "interval_interpretation",
                "eligible_batted_balls",
                "games",
            ]
        ].head(10)
    )
    print(qualified["interval_interpretation"].value_counts())

## 6. Component presentation (Phase 5)

Additive decomposition -- contact / unexplained residual / defensive
execution / advancement -- with status and reason codes. Provisional
components are NEVER implied to be equivalently validated to calibrated
ones, and there is no official leaderboard for any single component.

In [ ]:
from mlb_luck_score.scoring.public_score_table import describe_components

if public_score_table is not None and len(public_score_table) > 0:
    example_row = public_score_table.iloc[0]
    for component in describe_components(example_row):
        print(json.dumps(component, indent=2, default=str))

## 7. Development review tables (Phase 8, 2024 development only)

NOT permission to retune qualification thresholds or models based on how
these look.

In [ ]:
if review_tables is not None:
    print("Qualified intervals crossing zero:",
          review_tables["qualified_intervals_crossing_zero"]["count"],
          f"({review_tables['qualified_intervals_crossing_zero']['fraction_of_qualified']:.1%} of qualified)"
          if review_tables["qualified_intervals_crossing_zero"]["fraction_of_qualified"] is not None else "")
    display(pd.DataFrame(review_tables["widest_qualified_intervals"]).head(10))
    display(pd.DataFrame(review_tables["largest_provisional_component_shares"]).head(10))

In [ ]:
if review_tables is not None:
    comparison = pd.DataFrame(review_tables["ranking_with_vs_without_provisional_components"])
    display(comparison.sort_values("rank_delta", key=lambda s: s.abs(), ascending=False).head(10))

## 8. Single-digit-sample confirmation (Phase 8)

`mlb_luck_score.scoring.run_public_score.assert_no_single_digit_sample_
players_ranked` already raises if this is ever violated -- confirmed again
here descriptively.

In [ ]:
if public_score_table is not None:
    ranked = public_score_table["official_rank_favorable"].notna() | public_score_table["official_rank_unfavorable"].notna()
    single_digit = public_score_table["eligible_batted_balls"] < 10
    print("Single-digit-sample rows carrying an official rank:", int((ranked & single_digit).sum()))

## 9. Confirmation: 2025 untouched

In [ ]:
if score_card is not None:
    print("Seasons in this public score table:", score_card["seasons"])